# Multi-Period Portfolio Optimization Baseline

Driver for the deterministic EWMA + Ledoit-Wolf + ADMM baseline. The implementation lives in
`multi_period_admm.py` and is covered by `test_multi_period_admm.py`, so this notebook only
runs it and checks the output.

`LAG_driven_portfolio_oprimization.ipynb` is the original exploratory draft, kept for history.

In [ ]:
import numpy as np
import pandas as pd

from multi_period_admm import (
    HORIZON,
    TOLERANCE,
    admm_multi_period_optimizer,
    estimate_parameters,
    generate_synthetic_market_data,
)

pd.set_option('display.float_format', lambda value: f'{value:.6f}')

## 1. Data and parameter estimates

The synthetic generator is a deterministic smoke-test fixture, not a research dataset. Final
results need a documented real-data source.

In [ ]:
returns = generate_synthetic_market_data()
mu_sequence, sigma_sequence = estimate_parameters(returns)
mu_target = mu_sequence[-HORIZON:]
sigma_target = sigma_sequence[-HORIZON:]

print(f'returns: {returns.shape}')
print(f'parameter horizon: {mu_target.shape[0]} periods, {mu_target.shape[1]} assets')
print(f'minimum covariance eigenvalue: {min(np.linalg.eigvalsh(m).min() for m in sigma_target):.8f}')

## 2. Optimize and check

The assertions are the regression contract: convergence, simplex feasibility, and
diversification. A silent corner solution or an early stop fails here instead of printing a
passing summary.

In [ ]:
initial_weights = np.full(mu_target.shape[1], 1.0 / mu_target.shape[1])
weights, trades, residual_trace = admm_multi_period_optimizer(
    mu_target, sigma_target, initial_weights
)

assert np.isfinite(weights).all() and np.isfinite(trades).all()
assert np.all(weights >= -1e-10)
assert np.allclose(weights.sum(axis=1), 1.0, atol=1e-8)

final_primal, final_dual = residual_trace[-1]
assert final_primal < TOLERANCE and final_dual < TOLERANCE, (
    f'ADMM stopped without reaching tolerance after {len(residual_trace)} iterations: '
    f'primal={final_primal:.3e}, dual={final_dual:.3e}.'
)

effective_assets = 1.0 / np.sum(weights[-1] ** 2)
assert effective_assets > 2.0, (
    f'Portfolio collapsed to {effective_assets:.2f} effective assets; check the risk-aversion scale.'
)

summary = pd.Series({
    'periods_optimized': weights.shape[0],
    'assets': weights.shape[1],
    'iterations': len(residual_trace),
    'final_primal_residual': final_primal,
    'final_dual_residual': final_dual,
    'zero_trade_fraction': float(np.mean(np.abs(trades) <= 1e-6)),
    'final_active_positions': int(np.count_nonzero(weights[-1] > 1e-6)),
    'final_effective_assets': effective_assets,
})
print(summary)
print('Baseline checks passed; ADMM reached tolerance.')

## 3. Allocation over time

Weight of each held asset per period, plus realized turnover.

In [ ]:
allocation = pd.DataFrame(weights, columns=returns.columns)
held = allocation.loc[:, (allocation > 1e-4).any()]
turnover = pd.Series(np.abs(trades).sum(axis=1), name='turnover')

print(f'assets ever held above 1bp: {held.shape[1]} of {allocation.shape[1]}')
print(f'mean turnover per period: {turnover.mean():.6f}')
print('\nfinal allocation (descending):')
print(allocation.iloc[-1].sort_values(ascending=False).head(10))

## Open issues

`GAMMA` was chosen because it diversifies, not derived; it should target a volatility level.
`LAMBDA` is unrelated to real trading costs, so the zero-trade fraction is a property of the
penalty rather than a finding. Next milestone is a walk-forward backtest against equal-weight,
buy-and-hold, and single-period Markowitz, net of costs, on real returns.